# Lab 3: Build it, then try to break it
*Python for Healthcare · Stanford Technology Training (ITS-2519) · Dr. Lionel Levine*
**Session 2 · 55 minutes: about 45 minutes of work, then we debrief together**

Kestrel Valley Health runs a Diabetes Prevention Program (DPP). This year it has room for **20% of the patients**
in its prediabetes registry. You'll build a model to decide whom to invite first, then try to break it: look for a
leak, check that its risk numbers mean what they say, test it at a second site, and check whom it misses.

**The data** (all synthetic). Adults whose A1c at an index visit between January 2021 and September 2022 was
5.7 to 6.4%, followed for 3 years.
- `site_a.csv`: 5,000 patients at Kestrel Valley Health's original clinics. You'll train and test here.
- `site_b.csv`: 2,000 patients at four clinics Kestrel Valley Health acquired in 2024. They send their A1c
  tests to a different laboratory, and their patients are older. You'll test here.
- `data_dictionary.csv`: what each column means and when it was recorded.

The model would be used **at the index visit**, so it may only use what's known at that visit.

## Step 0. Load the data (1 minute)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/lmlevine/python-for-healthcare/main/data/session2/"

def load(name):
    """Read a course file: your uploaded copy if there is one, otherwise the course repo."""
    return pd.read_csv(name) if Path(name).exists() else pd.read_csv(DATA_URL + name)


site_a = load("site_a.csv")
site_b = load("site_b.csv")
dictionary = load("data_dictionary.csv")

print("Site A:", site_a.shape, "   Site B:", site_b.shape)
print(f"Developed type 2 diabetes within 3 years: Site A {site_a['t2d_3yr'].mean():.1%}, "
      f"Site B {site_b['t2d_3yr'].mean():.1%}")
site_a.head()

## Step 1. Read the data dictionary (2 minutes)
Which column is the outcome? Which one is only an identifier? Skim the rest; you'll come back to this table.

In [ ]:
dictionary

## Step 2. The vague request (8 minutes)
Open the Gemini panel and type exactly:
`Build a model that predicts t2d_3yr from the other columns in site_a, and tell me how good it is.`
Let it write and run its code. Don't correct it. Then fill in the table (double-click it to type).
If Gemini is still working after about 5 minutes, stop it and fill in what you can.

| Question | What Gemini did |
|---|---|
| Which columns did it use? |  |
| How did it split training and test data? |  |
| Which metric(s) did it report, and what values? |  |
| Did it warn you about any column? |  |

Now run the shared draft below, so the whole class has the same starting point. It reloads the data first, in case
Gemini's code changed `site_a`. It was written for this lab, and it does exactly what the request literally says:
it uses every other column. (A careful assistant may refuse to. If yours did, note how it decided, and whether you
could have checked its reason yourself.) If Gemini isn't available, this is also your fallback.

In [ ]:
site_a = load("site_a.csv")      # fresh copies, whatever the Gemini step changed
site_b = load("site_b.csv")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

X = site_a.drop(columns=["patient_id", "t2d_3yr"])
y = site_a["t2d_3yr"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

draft = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
draft.fit(X_train, y_train)

print(f"Accuracy: {accuracy_score(y_test, draft.predict(X_test)):.3f}")
print(f"AUROC:    {roc_auc_score(y_test, draft.predict_proba(X_test)[:, 1]):.3f}")

**Accuracy check.** A "model" that predicts *no one develops diabetes* is right for every patient who doesn't.
What accuracy would it get here? (About 22% of Site A develops diabetes.) Accuracy rewards the easy answer, so
for the rest of the lab we use **AUROC**: the chance that a randomly chosen future case gets a higher risk score
than a randomly chosen patient who stays diabetes-free (0.5 is a coin flip, 1.0 is perfect ranking).

## Step 3. The model zoo (3 minutes)
Six model families, one loop. Every model gets the same training rows and is scored on the same test rows.
Swapping the model is one line; the evaluation is the work. This cell takes 5 to 15 seconds.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

MODELS = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "K-nearest neighbors": KNeighborsClassifier(n_neighbors=50),
    # An SVM outputs scores; the wrapper turns them into probabilities.
    # (Older code writes SVC(probability=True), which scikit-learn 1.9 deprecates.)
    "Support vector machine": CalibratedClassifierCV(SVC(), ensemble=False),
    "Random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=42),
    "Neural network": MLPClassifier(hidden_layer_sizes=(16,), early_stopping=True, max_iter=500, random_state=42),
}

def split(features, data=site_a):
    """The same 75/25 train/test split every time, stratified on the outcome."""
    return train_test_split(data[features], data["t2d_3yr"], test_size=0.25,
                            stratify=data["t2d_3yr"], random_state=42)

def fit(model, X_train, y_train):
    """Fill blanks with the training median, standardize, then fit a fresh copy of the model."""
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clone(model)).fit(X_train, y_train)

def compare_models(features):
    """Fit every model on the same training rows and report its test AUROC, best first."""
    X_train, X_test, y_train, y_test = split(features)
    auroc = {name: roc_auc_score(y_test, fit(m, X_train, y_train).predict_proba(X_test)[:, 1])
             for name, m in MODELS.items()}
    return pd.Series(auroc, name="test AUROC").sort_values(ascending=False).round(3)

ALL_COLUMNS = [c for c in site_a.columns if c not in ("patient_id", "t2d_3yr")]
compare_models(ALL_COLUMNS)

1. Which family wins? How far apart are the six?
2. Is an AUROC near 0.97 believable for predicting diabetes 3 years ahead from one clinic visit?

## Step 4. The leak hunt (10 minutes)
A score that good usually means the model can see the answer. Find out how.

**4a. What does the model lean on?** Shuffle one column at a time in the test set and watch the AUROC drop.
A big drop means the model depends on that column.

In [ ]:
from sklearn.inspection import permutation_importance

def importance(features):
    """Shuffle one column at a time in the test set. How much does the AUROC drop?"""
    X_train, X_test, y_train, y_test = split(features)
    model = fit(LogisticRegression(max_iter=1000), X_train, y_train)
    result = permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=42)
    return pd.Series(result.importances_mean, index=features, name="AUROC drop").sort_values()

imp = importance(ALL_COLUMNS)
imp.plot.barh(figsize=(6, 4), title="What the model leans on (all columns)")
plt.xlabel("Drop in test AUROC when the column is shuffled")
plt.show()

**4b. When were the top columns recorded?** Look them up in the dictionary.

In [ ]:
top3 = imp.index[::-1][:3]          # the three columns the model leans on most
dictionary.set_index("column").loc[top3]

Think like a clinician: why would `metformin_started` predict a diabetes diagnosis so well?
Then check the `recorded` column: when was it read from the record?

**4c. Remove the top column only.** Is the score believable now?

In [ ]:
compare_models([c for c in ALL_COLUMNS if c != "metformin_started"])

**4d. Look again.**

In [ ]:
imp = importance([c for c in ALL_COLUMNS if c != "metformin_started"])
imp.plot.barh(figsize=(6, 4), title="What the model leans on (without metformin_started)")
plt.xlabel("Drop in test AUROC when the column is shuffled")
plt.show()

Why would the *number* of A1c tests predict diabetes better than the A1c value itself? When was it counted?

*Optional:* paste the data dictionary into Gemini and ask: `Which of these columns would not be known at the
index visit?` Compare its answer with yours.

## Step 5. Make the leak rule a refusal (4 minutes)
A leak you found once will come back the next time someone reruns the analysis with a new extract. So write the rule
into the notebook. This one is an *allow-list*: it accepts only columns the dictionary marks as known at the index
visit, so a new column gets refused until someone checks it. Run it on all the columns first.
**You'll see a red error. That's the refusal working.**

In [ ]:
KNOWN_AT_INDEX = dictionary.loc[dictionary["recorded"] == "index visit", "column"].tolist()

def refuse_leaks(features):
    """The leak rule, written as a refusal: every feature must be known at the index visit."""
    unknown = [f for f in features if f not in KNOWN_AT_INDEX]
    if unknown:
        raise ValueError(f"Refusing to fit: {unknown} aren't known at the index visit.")
    print(f"PASS  all {len(features)} features are known at the index visit")

refuse_leaks(ALL_COLUMNS)

The refusal is the point: this notebook now can't fit a model on those columns. Fill in `LEAKS` below, then run
the cell. It refuses until your list is right, and then reruns the zoo on honest features. The later steps check
the rule again, so they refuse too until `LEAKS` is right.

In [ ]:
LEAKS = []   # TODO: list the columns that weren't known at the index visit
HONEST = [c for c in ALL_COLUMNS if c not in LEAKS]

refuse_leaks(HONEST)
compare_models(HONEST)

Which family wins now? By how much? Was the fancy model worth it?

## Step 6. Will it work in the program? (7 minutes)
**6a. The capacity table.** The DPP can take the top 20% by predicted risk. Which future cases does it reach?

In [ ]:
refuse_leaks(HONEST)                       # the rule runs again before anything is reported
X_train, X_test, y_train, y_test = split(HONEST)
model = fit(LogisticRegression(max_iter=1000), X_train, y_train)
risk = model.predict_proba(X_test)[:, 1]

def capacity_table(risk, y, shares=(0.10, 0.20, 0.30)):
    """If the program can take the top share of patients by risk, whom does it reach?"""
    y = np.asarray(y)
    rows = []
    for share in shares:
        cutoff = np.quantile(risk, 1 - share)
        offered = risk >= cutoff
        rows.append({
            "program size": f"top {share:.0%}",
            "risk cutoff": round(cutoff, 2),
            "patients offered": int(offered.sum()),
            "future cases among them": int(y[offered].sum()),
            "PPV (share of those offered)": f"{y[offered].mean():.0%}",
            "sensitivity (share of all future cases)": f"{y[offered].sum() / y.sum():.0%}",
        })
    return pd.DataFrame(rows).set_index("program size")

print(f"Test set: {len(y_test)} patients, {y_test.sum()} future cases. "
      f"Honest AUROC {roc_auc_score(y_test, risk):.3f}")
capacity_table(risk, y_test)

1. With room for 20%, what share of all future cases does the program reach (sensitivity)?
2. Of the patients offered a place, what share would develop diabetes without it (PPV)?
3. If the program picked the same number of patients at random, roughly what share of future cases would it reach?

**6b. Calibration.** When the model says 30%, do about 30% develop diabetes?

In [ ]:
def calibration_table(risk, y, groups=10):
    """Sort patients into tenths by predicted risk, then compare predicted and observed rates."""
    t = pd.DataFrame({"predicted": risk, "observed": np.asarray(y)})
    t["tenth"] = pd.qcut(t["predicted"], groups, labels=False, duplicates="drop") + 1
    return t.groupby("tenth")[["predicted", "observed"]].mean()

def plot_calibration(tables, title):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    for label, t in tables.items():
        ax.plot(t["predicted"], t["observed"], marker="o", label=label)
    ax.plot([0, 0.7], [0, 0.7], linestyle="--", color="gray", label="perfect calibration")
    ax.set_xlabel("Predicted risk (mean in each tenth)")
    ax.set_ylabel("Observed rate")
    ax.set_title(title)
    ax.legend(frameon=False)
    plt.show()

cal_a = calibration_table(risk, y_test)
plot_calibration({"Site A test set": cal_a}, "Calibration at Site A")
cal_a.round(2).T

## Step 7. Try it at Site B (5 minutes)
The acquired clinics want to use the same model. Run it on Site B, unchanged.

In [ ]:
risk_b = model.predict_proba(site_b[HONEST])[:, 1]      # the Site A model, unchanged, applied to Site B

print(f"AUROC:               Site A test {roc_auc_score(y_test, risk):.3f}   Site B {roc_auc_score(site_b['t2d_3yr'], risk_b):.3f}")
print(f"Mean predicted risk: Site A test {risk.mean():.1%}   Site B {risk_b.mean():.1%}")
print(f"Observed rate:       Site A test {y_test.mean():.1%}   Site B {site_b['t2d_3yr'].mean():.1%}")

plot_calibration({"Site A test set": cal_a, "Site B": calibration_table(risk_b, site_b["t2d_3yr"])},
                 "Calibration: Site A vs Site B")

1. Did the **ranking** hold up (AUROC)?
2. Did the **risk numbers** hold up (calibration)? Are the Site B points above or below the diagonal?
3. Site B sends A1c to a different laboratory, and its patients are older. Which of those could make the model
   wrong at Site B, and in which direction?
4. If Site B only ranks its own patients for a fixed number of places, does the calibration problem matter?
   What if a clinician quotes the risk number to a patient?

## Step 8. Whom does it miss? (3 minutes)
Same test set, same 20% cutoff, split by age.

In [ ]:
cutoff_20 = np.quantile(risk, 0.80)          # the top-20% cutoff from the capacity table
test = X_test.assign(future_case=y_test.to_numpy(), risk=risk, offered=risk >= cutoff_20)
test["age group"] = np.where(test["age"] < 45, "under 45", "45 and over")

by_age = test.groupby("age group").agg(
    patients=("future_case", "size"),
    observed_rate=("future_case", "mean"),
    mean_predicted=("risk", "mean"),
    share_offered=("offered", "mean"),
    fpg_missing=("fpg", lambda s: s.isna().mean()),
)
by_age["future cases reached"] = test[test["future_case"] == 1].groupby("age group")["offered"].mean()
by_age.round(2)

1. Do patients under 45 develop diabetes less often than older patients?
2. What share of their future cases does the program reach? Of older patients' future cases?
3. Look at `fpg_missing`. Why would a missing lab value push someone *down* the list?

## Step 9. Write the do-not-use rules (2 minutes)
Write three rules for this model, each one specific enough that someone could check it (double-click the cell
below to type). Start each with *Don't use this model…* or *Stop using it if…*.

1.
2.
3.

## Before the debrief, jot down
1. What gave the leak away first: the score, the importance chart, or the dictionary?
2. Which result would you show the DPP manager, and which one would you show Site B's medical director?
3. One thing this model must refuse to do.